# Détection automatique d'erreurs d'OCR

## Imports

In [1]:
import nltk
from thefuzz import fuzz
from thefuzz import process
from Levenshtein import distance

Nous allons d'abord utiliser la distance de Levenshtein, selon sa formule originale (1965)

In [2]:
distance("bulletin", "ballotins")

3

Nous allons ensuite faire appel à la librairie TheFuzz (anciennement fuzzywuzzy) qui repose sur python-Levenshtein (https://github.com/seatgeek/thefuzz).
Celle-ci permet de normaliser le résultat en calculant un ratio (pourcentage) de similarité entre deux chaînes de caractères :

In [3]:
fuzz.ratio("bulletin", "ballotins")

71

La fonction  `partial_ratio` se focalise sur la sous-chaîne (substring) la plus similaire : 

In [4]:
fuzz.partial_ratio("Grand", "Grand-Place")

100

La fonction `token_sort_ratio` ignore l'ordre des mots :

In [5]:
fuzz.token_sort_ratio("archives de la ville de Bruxelles", "de Bruxelles la ville de archives")

100

La fonction `token_set_ratio` ignore les répétitions de mots :

In [6]:
fuzz.token_set_ratio("plus jamais ça !", "plus jamais Jamais JAMAIS ça !!!")

100

Chargeons à présent le fichier `all.txt` (tronqué) : 

In [ ]:
data_path = "../data/all.txt"

# Lire seulement les 50M premiers caractères
LIMIT = 50000000

# Augmenter le timeout à 60 secondes (par défaut : 5 secondes !)
import nltk.redos
nltk.redos.DEFAULT_TIMEOUT = 60.0

text = open(data_path, "r", encoding="utf-8").read()[:LIMIT]

tokens = nltk.wordpunct_tokenize(text)
print(f"{len(tokens)} words (tokens) found")

vocab = set(tokens)
print(f"{len(vocab)} different word forms")

12744515 words (tokens) found
777629 different word forms


Nous pouvons identifier les variantes d'un mot par ordre décroissant de similarité : 

In [8]:
word = "bruxelles"
process.extract(word, vocab, limit=15)

[('Bruxelles', 100),
 ('BRUXELLES', 100),
 ('BRUXEllES', 100),
 ('bruxelles', 100),
 ('Baruxelles', 95),
 ('BRUXJELLES', 95),
 ('BruxellesJ', 95),
 ('iBruxelles', 95),
 ('àgBruxelles', 95),
 ('SBRUXELLES', 95),
 ('BRUNXELLES', 95),
 ('Brutxelles', 95),
 ('eBRUXELLES', 95),
 ('Bruxelle', 94),
 ('Bruxellés', 94)]

Les caractères accentués ne sont pas pris en compte pour le calcul du score, mais cela reste une bonne base pour récupérer les occurrences mal OCRisées d'un mot.